# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Clustering**.

Per the task-type table: "What kinds of items exist?" → Clustering, target = none, metric = silhouette + human sense-check.

That's exactly the question I'm asking — not "will this page decline" (that would be classification, needing an observed future outcome) and not "which page first" (that would be ranking).

I'm asking what recurring types of pages exist in the inventory, based on structured metrics only, so clustering is the honest fit — not classification dressed up, and not "semantic clustering," since there's no article text in this dataset.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**None — this is unsupervised, and that's the honest answer, not a gap to fill.**

Clustering doesn't predict a label; it groups pages by similarity across features like `sessions_90d`, `engagement_rate`, `trend_pct`, `ctr`, `avg_position`, `ai_traffic_pct`, `content_age_days`, and `days_since_last_update`.

If you want to be extra careful here (worth a line in your write-up): the archetype names I apply afterward (hidden gem, stale visible, etc.) are not targets — they're labels I assign to clusters only after inspecting what's actually in them, per the "don't name before inspecting" warning. They're descriptive labels for existing groups, not predictions of a future or hidden truth.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Silhouette score, backed by a human sense-check.** One number I can defend: silhouette score (ranges -1 to 1, higher means clusters are dense and well-separated). It's the standard metric for K-Means when there's no ground truth to compare against.

But — say this explicitly in your write-up — silhouette alone can be gamed by degenerate clusters (e.g. one giant cluster + tiny outlier clusters can still look "good" numerically). So the real bar for "good" is: does the cluster profile match a real, recognizable archetype from your list (champion, hidden gem, stale visible, etc.) when you look at the actual median values per cluster? A silhouette score with clusters that make no business sense isn't a win.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one page (`content_id`), scoped to one client. Here's the code to show it:

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# unit of analysis: one row = one content page
cols_to_show = [
    "content_id", "client_id", "content_type",
    "sessions_90d", "engagement_rate", "trend_direction",
    "trend_pct", "impressions_90d", "ctr", "avg_position",
    "content_age_days", "days_since_last_update"
]
print(df[cols_to_show].head(5))
print("\nOne row =", "one content page, scoped to one client_id")
print("Total rows:", len(df), "| Total unique pages:", df["content_id"].nunique())

             content_id          client_id     content_type  sessions_90d  \
0  content_304f48230142  client_f369cb89fc  keyword article            17   
1  content_a1fb4e703a9e  client_4e07408562  keyword article             9   
2  content_9aa793d4d895  client_7f2253d7e2  keyword article            11   
3  content_331d6c4de07b  client_19581e27de  keyword article            78   
4  content_d99b7a2d90ca  client_3fdba35f04  keyword article           145   

   engagement_rate trend_direction  trend_pct  impressions_90d   ctr  \
0             5.88            down      -41.4             3803  0.76   
1             0.00            down      -57.7            15320  0.05   
2             0.00            down      -60.9            12581  0.09   
3             1.28          stable      -13.8            11751  0.49   
4             0.00            down      -34.7            19140  0.13   

   avg_position  content_age_days  days_since_last_update  
0          10.6               187           

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The archetypes aren't defined by any single threshold — they emerge from combinations of signals that pull in different directions:

* A page can look "bad" on traffic alone (bottom 10% impressions) but still be a **hidden gem** if engagement is high — a single traffic rule would prune it wrongly.
* 54% of pages are trending down, but that alone doesn't tell you if a page is a **stale visible page** (still gets impressions, worth rescuing) or a **weak/no-demand** page (never had traffic to lose) — you need trend and volume and engagement together.
* **Cannibalization-risk** pages can't
be spotted from any single page's metrics at all — you'd need to notice similar pages competing for the same intent, which is inherently a multi-signal, relative comparison, not an if-statement on one row.

A hand-written rule would need a tangle of nested conditions across 10+ correlated features, and any two analysts would draw the thresholds differently. Clustering lets the structure in the data set the groupings instead of a guessed cutoff — that's the whole justification for using it over a plain rule here.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.